# Mini RAG — 학습 및 구현 테스트 기록

이 노트북은 **Cosine Similarity → Retrieval → Chunking → Metadata → LLM Generation → RAG 유무 비교** 순서로 공부하면서 직접 작성한 코드와 주석을 보존한 기록입니다. 학습 과정에서 같은 함수를 여러 번 수정하거나 재정의한 부분도 흐름을 이해할 수 있도록 남겼습니다.

**연관된 GitHub 파일** (Colab의 상대 경로가 아닌 실제 저장소 링크):

- [최종 Python 구현](https://github.com/Lymonada/research-prep/blob/main/pytorch_basics/13_mini_RAG.py)
- [Retrieval 정리 노트](https://github.com/Lymonada/research-prep/blob/main/notes/11_Mini%20RAG%20Retrieval%20Pipeline.md)
- [Augmentation / Generation 정리 노트](https://github.com/Lymonada/research-prep/blob/main/notes/12_Mini%20RAG%20Augmentation%20and%20Generation.md)
- [실험에 사용한 학습 노트 폴더](https://github.com/Lymonada/research-prep/tree/main/notes)
- [RAG ON/OFF 실험용 문서](https://github.com/Lymonada/research-prep/blob/main/notes/rag_test.md)

**당시 실행 환경:** Google Colab에서 Google Drive에 위치한 `research-prep` 저장소 사본으로 이동한 후, GitHub의 최신 파일을 `git pull`하여 `notes/*.md`를 읽었습니다. 아래 원래의 `%cd` 경로는 **작성자의 Google Drive 작업 경로**이므로, 다른 환경에서 재실행하려면 사용자가 먼저 자신의 저장소 경로를 준비해야 합니다. 자동 `git clone` 코드나 새 파일 로딩 코드는 추가하지 않았습니다.

> 학습 당시의 출력은 이전 실행 결과입니다. 저장된 API 응답은 현재 서비스 동작을 보증하지 않으며, 외부 LLM/API 호출을 이번에 다시 실행하지 않았습니다.


In [ ]:
# 작업 폴더로 이동
%cd /content/drive/MyDrive/Github/research-prep

!git pull origin main

## Cosine Similarity

In [ ]:
import torch

query = torch.tensor([1.0, 2.0, 3.0])

doc1 = torch.tensor([1.0, 2.0, 3.0])
doc2 = torch.tensor([3.0, 2.0, 1.0])
doc3 = torch.tensor([-1.0, -2.0, -3.0])

def cosine_similarity(a, b):
    # 직접 구현
    dot = torch.dot(a,b)
    norm_a = torch.norm(a)
    norm_b = torch.norm(b)
    return dot / (norm_a * norm_b)

print(cosine_similarity(query, doc1))
print(cosine_similarity(query, doc2))
print(cosine_similarity(query, doc3))


documents = torch.stack([doc1, doc2, doc3])

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores

scores = cosine_similarity_batch(query, documents)

print(scores)
print(scores.shape)




tensor(1.0000)
tensor(0.7143)
tensor(-1.0000)
tensor([ 1.0000,  0.7143, -1.0000])
torch.Size([3])


## Retriever

In [ ]:
import torch

query = torch.tensor([1.0, 2.0, 3.0])

doc1 = torch.tensor([1.0, 2.0, 3.0])
doc2 = torch.tensor([3.0, 2.0, 1.0])
doc3 = torch.tensor([-1.0, -2.0, -3.0])
documents = torch.stack([doc1, doc2, doc3])

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores

def retrieve(query, documents, texts, k=2):
    # 1. cosine similarity 계산
    # scores: [N]
    scores = cosine_similarity_batch(query, documents)

    # 2. top-k score와 index 선택
    # top_scores: [k]
    # top_indices: [k]
    top_scores, top_indices = torch.topk(scores, k)

    # 3. results = [] 생성
    results = []

    # 4. top_scores와 top_indices를 순회
    #    실제 text와 score를 묶어서 results에 추가
    for score, idx in zip(top_scores, top_indices):
      index = idx.item()
      results.append((texts[index], score.item()))

    # 5. results 반환
    return results

texts = [
    "Transformers use attention mechanisms.",
    "RAG retrieves relevant documents before generation.",
    "Cats are popular household pets."
]

results = retrieve(
    query,
    documents,
    texts,
    k=2
)

print(results)

[('Transformers use attention mechanisms.', tensor(1.0000)), ('RAG retrieves relevant documents before generation.', tensor(0.7143))]


## Testing Embedding Model

In [ ]:
!pip install sentence-transformers

In [ ]:
import torch
from sentence_transformers import SentenceTransformer


texts = [
    "Transformers use attention mechanisms.",
    "RAG retrieves relevant documents before generation.",
    "Cats are popular household pets.",
    "Vector databases store embedding vectors.",
    "PyTorch is a deep learning framework."
]
query_text = "How does RAG find useful information?"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
) # D = 384

document_embeddings = model.encode(texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(document_embeddings.shape)
print(query_embedding.shape)

print(document_embeddings[0][:10])
print(query_embedding[:10])

torch.Size([5, 384])
torch.Size([384])
tensor([-1.4258e-02,  1.4868e-02, -1.8246e-02, -1.9041e-02,  2.7812e-03,
        -8.4366e-05,  9.8082e-03,  6.4557e-02,  6.8258e-02,  2.3202e-02],
       device='cuda:0')
tensor([-0.0235,  0.0882,  0.0186,  0.0019, -0.0564,  0.0139,  0.0368,  0.0518,
        -0.0119, -0.0019], device='cuda:0')


## Semantic Retriever Test

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

texts = [
    "Transformers use attention mechanisms.",
    "RAG retrieves relevant documents before generation.",
    "Cats are popular household pets.",
    "Vector databases store embedding vectors.",
    "PyTorch is a deep learning framework."
]
query_text = "How does RAG find useful information?"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
) # D = 384

document_embeddings = model.encode(texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores

def retrieve(query, documents, texts, k=2):

    scores = cosine_similarity_batch(query, documents)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):
      index = idx.item()
      results.append((texts[index], score.item()))

    return results


results = retrieve(
    query_embedding,
    document_embeddings,
    texts,
    k=2
)

print(results)

[('RAG retrieves relevant documents before generation.', 0.6222127079963684), ('Transformers use attention mechanisms.', 0.2505756914615631)]


## Chunking

In [ ]:
def chunk_text(text, chunk_size):
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results

text = "A B C D E F G H I J"
chunks = chunk_text(text, chunk_size=4) # chunks: list[str]

print(chunks)

['A B C D', 'E F G H', 'I J']


## Chunking with Overlap

In [ ]:
def chunk_text_overlap(text, chunk_size, overlap):
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results

text = "A B C D E F G H I J"
chunks = chunk_text_overlap(text, chunk_size=4, overlap=2) # chunks: list[str]

print(chunks)

['A B C D', 'C D E F', 'E F G H', 'G H I J', 'I J']


## Embedding Model with Chunking

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores # scores: [N]

def retrieve(query_embedding, chunk_embeddings, texts, k=2): # query_embedding: [D], chunk_embeddings: [N, D]

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings) # scores: [N]
    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):
      index = idx.item()
      results.append((texts[index], score.item()))

    return results

def chunk_text_overlap(text, chunk_size, overlap):
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results

document = """
Transformers are neural network architectures that use attention mechanisms
to model relationships between tokens in a sequence.

Self-attention allows each token to examine other tokens in the sequence
and build a contextual representation.

Retrieval-Augmented Generation, or RAG, combines information retrieval
with language generation. Before answering a question, a RAG system searches
an external collection of documents for relevant information.

Documents in a RAG system are usually split into smaller chunks.
Each chunk is converted into an embedding vector using an embedding model.
These vectors can then be searched using similarity measures such as cosine similarity.

When a user submits a query, the query is also converted into an embedding.
The system compares the query embedding with the stored chunk embeddings
and retrieves the most relevant chunks.

The retrieved chunks are placed into the prompt as additional context.
A language model then uses both the user question and the retrieved context
to generate an answer.
"""
query_text = "How does a RAG system find relevant information?"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
) # D = 384

chunks = chunk_text_overlap(document, chunk_size=35, overlap=8) # chunks: list[str]

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print()

chunks_embeddings = model.encode(chunks, convert_to_tensor=True) # shape: [N,D]
query_embedding = model.encode(query_text, convert_to_tensor=True) # shape: [D]

results = retrieve(query_embedding, chunks_embeddings, chunks, k=2)
# texts 자리에 "각 embedding row와 1:1로 대응되는 원본 문자열 목록"이 들어가야 해서 chunks가 들어감.
# 즉 chunks로 chunks_embedding을 만들었기 때문에 chunks_embeddings와 chunks가 들어감.

for text, score in results:
    print(f"Score: {score:.4f}")
    print(text)
    print()


Chunk 0:
Transformers are neural network architectures that use attention mechanisms to model relationships between tokens in a sequence. Self-attention allows each token to examine other tokens in the sequence and build a contextual representation. Retrieval-Augmented Generation,

Chunk 1:
sequence and build a contextual representation. Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation. Before answering a question, a RAG system searches an external collection of documents for relevant information. Documents in a

Chunk 2:
of documents for relevant information. Documents in a RAG system are usually split into smaller chunks. Each chunk is converted into an embedding vector using an embedding model. These vectors can then be searched using

Chunk 3:
model. These vectors can then be searched using similarity measures such as cosine similarity. When a user submits a query, the query is also converted into an embedding. The system compares the 

## Loading Document with Metadata

In [ ]:
from pathlib import Path


def load_document(file_path): # ex) data/transformer_decoder.md

  # 1. file_path를 Path 객체로 변환
  path = Path(file_path)

  # 2. 파일 내용을 UTF-8로 읽기
  text = path.read_text(encoding="utf-8")

  # 3. document dictionary 생성
  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict

## Chunking with Metadata

In [ ]:
def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    # 1. document의 text를 기존 chunk_text()로 나누기
    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    # 2. 최종 chunk들을 담을 빈 리스트
    chunks_with_metadata = []

    # 3. 각 text chunk를 순회
    for i, chunk in enumerate(text_chunks):

        # document의 metadata를 복사
        metadata = document["metadata"].copy()

        # metadata에 chunk_index 추가
        metadata["chunk_index"] = i

        # text + metadata를 담은 dictionary 생성
        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }

        # chunks_with_metadata에 append
        chunks_with_metadata.append(chunk_dict)
    # 4. 반환
    return chunks_with_metadata

chunks = chunk_document(document, chunk_size=100, overlap=20)
print(chunks[0])

## Loading and Chunking with Github


In [ ]:
from pathlib import Path


def load_document(file_path): # ex) data/transformer_decoder.md

  # 1. file_path를 Path 객체로 변환
  path = Path(file_path)

  # 2. 파일 내용을 UTF-8로 읽기
  text = path.read_text(encoding="utf-8")

  # 3. document dictionary 생성
  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    # 1. document의 text를 기존 chunk_text()로 나누기
    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    # 2. 최종 chunk들을 담을 빈 리스트
    chunks_with_metadata = []

    # 3. 각 text chunk를 순회
    for i, chunk in enumerate(text_chunks):

        # document의 metadata를 복사
        metadata = document["metadata"].copy()

        # metadata에 chunk_index 추가
        metadata["chunk_index"] = i

        # text + metadata를 담은 dictionary 생성
        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }

        # chunks_with_metadata에 append
        chunks_with_metadata.append(chunk_dict)
    # 4. 반환
    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

document = load_document("notes/07_attention.md") # 문서 내용과 metadata를 둘다 가지는 dict를 리턴
print(document["metadata"])
print(document["text"][:300])

# chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴
chunks = chunk_document(document, chunk_size=100, overlap=20)

print("전체 chunk 개수:", len(chunks))

print("\n첫 번째 chunk:")
print(chunks[0])

print("\n두 번째 chunk metadata:")
print(chunks[1]["metadata"])

print("\n첫 번째 chunk text:")
print(chunks[0]["text"])

{'source': '07_attention.md', 'path': 'notes/07_attention.md'}
# Attention: From Encoder-Decoder Attention to Multi-Head Self-Attention

이 문서는 RNN 계열 학습 이후 Attention을 처음 접했을 때부터  
**Encoder-Decoder Attention → Q/K/V → Self-Attention → Single-Head → Multi-Head Attention**으로 이어진 학습 흐름을 정리한 노트이다.

단순히 공식만 외우는 것이 아니라,

- Attention이 왜 필요했는지
- 각 계산이 어떤 의미를 가지는지
- Que
{'text': '# Attention: From Encoder-Decoder Attention to Multi-Head Self-Attention 이 문서는 RNN 계열 학습 이후 Attention을 처음 접했을 때부터 **Encoder-Decoder Attention → Q/K/V → Self-Attention → Single-Head → Multi-Head Attention**으로 이어진 학습 흐름을 정리한 노트이다. 단순히 공식만 외우는 것이 아니라, - Attention이 왜 필요했는지 - 각 계산이 어떤 의미를 가지는지 - Query / Key / Value가 왜 등장하는지 - Self-Attention에서는 무엇이 달라지는지 - Multi-Head Attention은 왜 필요한지 - 실제 tensor shape은 어떻게 변하는지 를 단계적으로 이해하는 것을 목표로 한다. --- ## 1. Why Attention? Attention을 이해하려면 먼저 초기 **Seq2Seq Encoder-Decoder RNN** 구조의 한계를 볼 필요가 있다. 기본적인 Seq2Seq에서는 Encoder가 입력 sequence를', 'metadata': {'source': '07_attention.md', 'path': 'not

## Multiple Files to a Chunk Corpus

In [ ]:
from pathlib import Path


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

print("전체 chunk 개수:", len(all_chunks))

print("\n첫 번째 chunk:")
print(all_chunks[0])

print("\n마지막 chunk:")
print(all_chunks[-1])

for file_path in file_paths:
    document = load_document(file_path)
    chunks = chunk_document(document, 100, 20)

    print(chunks[0]["metadata"])

전체 chunk 개수: 153

첫 번째 chunk:
{'text': '# Attention: From Encoder-Decoder Attention to Multi-Head Self-Attention 이 문서는 RNN 계열 학습 이후 Attention을 처음 접했을 때부터 **Encoder-Decoder Attention → Q/K/V → Self-Attention → Single-Head → Multi-Head Attention**으로 이어진 학습 흐름을 정리한 노트이다. 단순히 공식만 외우는 것이 아니라, - Attention이 왜 필요했는지 - 각 계산이 어떤 의미를 가지는지 - Query / Key / Value가 왜 등장하는지 - Self-Attention에서는 무엇이 달라지는지 - Multi-Head Attention은 왜 필요한지 - 실제 tensor shape은 어떻게 변하는지 를 단계적으로 이해하는 것을 목표로 한다. --- ## 1. Why Attention? Attention을 이해하려면 먼저 초기 **Seq2Seq Encoder-Decoder RNN** 구조의 한계를 볼 필요가 있다. 기본적인 Seq2Seq에서는 Encoder가 입력 sequence를', 'metadata': {'source': '07_attention.md', 'path': 'notes/07_attention.md', 'chunk_index': 0}}

마지막 chunk:
{'text': '따로 이해했으므로 다음 단계에서는 ```text Source Token IDs ↓ Encoder ↓ Encoder Final Output ↓ Decoder ↓ Final Decoder Representations ↓ Vocabulary Linear ↓ Logits / Loss ``` 를 **Transformer 전체 forward pass 하나의 흐름**으로 연결해서 보는 것이 자연스럽다. 그 이후에는 이 전체 구조를 PyTorch tensor shape과 작은 `.py` 구현으로 

## Retrieval with Documents

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores

def retrieve(query_embedding, chunk_embeddings, texts, k=2): # query_embedding: [D], chunk_embeddings: [N, D]

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings) # scores: [N]
    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):
      index = idx.item()
      results.append((texts[index], score.item()))

    return results

def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "How does attention make each token to contextual representation?"


chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

document_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(document_embeddings.shape)
print(query_embedding.shape)

results = retrieve(
    query_embedding,
    document_embeddings,
    chunk_texts,
    k=2
)

print(results)

torch.Size([153, 384])
torch.Size([384])
[('target position 특정 마지막 token에서만 source를 보는 것이 아니다. ### Decoder is Q, Encoder is K/V in Cross-Attention Decoder는 현재 필요한 source 정보를 찾는 쪽이고, Encoder는 검색 대상과 가져올 정보를 제공하는 쪽이다. ### Causal Mask blocks attention connections, not tokens themselves 미래 token을 sequence에서 삭제하는 것이 아니라 attention score에서 미래 위치를 볼 수 없도록 막는다. ### A token can attend to itself 현재 위치의 input token은 다음 token을 예측하기 위한 context이므로 자기 자신까지 볼 수 있다. ### Encoder final output is reused by every Decoder layer 각 Decoder layer의 Cross-Attention은 같은 Encoder final representations를 참고하지만, layer별 projection parameter는 서로 다르다.', 0.5619419813156128), ('Query로 다른 모든 token의 Key와 비교하고, > 해당 Value들을 가중합하면서 각 token을 문맥이 반영된 contextual representation으로 바꾼다. ### Multi-Head Attention > 같은 입력을 서로 다른 learned projection space에서 여러 번 바라보면서 > 서로 다른 attention pattern과 관계를 병렬로 학습한다. 따라서 전체적인 발전 흐름은, ```text RNN Seq2Seq ↓ Fixed context bottleneck ↓ Encoder-Decoder Attention ↓ Query / Key / Value ↓ Self-Attention ↓

## Metadata-aware Retrieval

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores

def retrieve(query_embedding, chunk_embeddings, texts, k=2): # query_embedding: [D], chunk_embeddings: [N, D]

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings) # scores: [N]
    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):
      index = idx.item()
      results.append((texts[index], score.item()))

    return results

def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list

In [ ]:
def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기
        # {
        #     "text": ...,
        #     "score": ...,
        #     "metadata": ...
        # }
        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "How does attention make each token to contextual representation?"

chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

document_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(document_embeddings.shape)
print(query_embedding.shape)

results = retrieve(
    query_embedding,
    document_embeddings,
    all_chunks,
    k=2
)

def print_results(results):

    for i, result in enumerate(results):

        print(f"\n=== Result {i + 1} ===")
        print(f"Score: {result['score']:.4f}")
        print(f"Source: {result['metadata']['source']}")
        print(f"Chunk index: {result['metadata']['chunk_index']}")
        print(f"Text: {result['text']}")

print_results(results)

torch.Size([153, 384])
torch.Size([384])

=== Result 1 ===
Score: 0.5619
Source: 09_transformer_decoder.md
Chunk index: 41
Text: target position 특정 마지막 token에서만 source를 보는 것이 아니다. ### Decoder is Q, Encoder is K/V in Cross-Attention Decoder는 현재 필요한 source 정보를 찾는 쪽이고, Encoder는 검색 대상과 가져올 정보를 제공하는 쪽이다. ### Causal Mask blocks attention connections, not tokens themselves 미래 token을 sequence에서 삭제하는 것이 아니라 attention score에서 미래 위치를 볼 수 없도록 막는다. ### A token can attend to itself 현재 위치의 input token은 다음 token을 예측하기 위한 context이므로 자기 자신까지 볼 수 있다. ### Encoder final output is reused by every Decoder layer 각 Decoder layer의 Cross-Attention은 같은 Encoder final representations를 참고하지만, layer별 projection parameter는 서로 다르다.

=== Result 2 ===
Score: 0.5554
Source: 07_attention.md
Chunk index: 40
Text: Query로 다른 모든 token의 Key와 비교하고, > 해당 Value들을 가중합하면서 각 token을 문맥이 반영된 contextual representation으로 바꾼다. ### Multi-Head Attention > 같은 입력을 서로 다른 learned projection space에서 여러 번 바라보면서 > 서로 다른 attention pattern과 관계를 병렬로 

# Text Formatting

### Text Formatting — 당시 문법 오류가 발생한 초안

아래는 학습 중 작성한 **원래 코드**입니다. 이 시점에 문법 오류가 발생했고, 뒤의 `Retrieval with Formatting Test`에서 수정한 버전으로 계속 진행했습니다. 기록을 위해 코드와 주석을 남기되 재실행 시 오류가 나지 않도록 텍스트로만 표시합니다.

```python
def format_context(retrieved_chunks):
  formatted_list = []
  for i, chunk in enumerate(retrieved_chunks):
    text = chunk["text"]
    metadata = chunk["metadata"]
    source = chunk["metadata"]["source"]
    chunk_index = chunk["metadata"]["chunk_index"]
    formatted_text = (
                f"[{i}]\n"
                f"Source: {source}\n"
                f"Chunk: {chunk_index}\n"
                f"Text: {text}"
            )

    formatted_list.append(formatted_text)

  result = "\n\n".join(formatted_list)
  return result

def build_prompt(question, context):

  prompt = f"""
    Answer the question using the provided context.

    If the context does not contain enough information,
    say that the information is not available in the context.

  Context:
  {context}

  Question:
  {question}

  Answer: """
  return prompt

  retrieved = retrieve(
    query_embedding,
    chunk_embeddings,
    all_chunks,
    k=3
)

context = format_context(retrieved)

print(context)
```

## Retrieval with Formatting Test

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list


def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기

        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results



def format_context(retrieved_chunks):
    formatted_list = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        text = chunk["text"]
        metadata = chunk["metadata"]

        source = metadata["source"]
        chunk_index = metadata["chunk_index"]

        formatted_text = (
            f"[{i}]\n"
            f"Source: {source}\n"
            f"Chunk: {chunk_index}\n"
            f"Text: {text}"
        )

        formatted_list.append(formatted_text)

    context = "\n\n".join(formatted_list)

    return context

def build_prompt(question, context):
    prompt = (
        "Answer the question using the provided context.\n\n"
        "If the context does not contain enough information, "
        "say that the information is not available in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    return prompt

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "How does attention turn each token into a contextual representation?"

chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(chunk_embeddings.shape)
print(query_embedding.shape)


retrieved = retrieve(
    query_embedding,
    chunk_embeddings,
    all_chunks,
    k=3
)

context = format_context(retrieved)

prompt = build_prompt(
    query_text,
    context
)

print(prompt)

torch.Size([153, 384])
torch.Size([384])
Answer the question using the provided context.

If the context does not contain enough information, say that the information is not available in the context.

Context:
[1]
Source: 07_attention.md
Chunk: 40
Text: Query로 다른 모든 token의 Key와 비교하고, > 해당 Value들을 가중합하면서 각 token을 문맥이 반영된 contextual representation으로 바꾼다. ### Multi-Head Attention > 같은 입력을 서로 다른 learned projection space에서 여러 번 바라보면서 > 서로 다른 attention pattern과 관계를 병렬로 학습한다. 따라서 전체적인 발전 흐름은, ```text RNN Seq2Seq ↓ Fixed context bottleneck ↓ Encoder-Decoder Attention ↓ Query / Key / Value ↓ Self-Attention ↓ Scaled Dot-Product Attention ↓ Multi-Head Attention ``` 으로 정리할 수 있다. --- ## 25. Where This Note Stops 이 문서는 **Multi-Head Self-Attention의 output을 만드는 과정까지**를 다룬다. 다음 단계에서는 이 Multi-Head Attention output이 Transformer Encoder Block 안에서 어떻게 사용되는지

[2]
Source: 09_transformer_decoder.md
Chunk: 41
Text: target position 특정 마지막 token에서만 source를 보는 것이 아니다. ### Decoder is Q, Encoder is K/V in Cross-Att

## LLM Generation

In [ ]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def generate_answer(prompt): # prompt는 format_context를 거쳐서 나온 retrieved를 다시 build_prompt로 다듬어서 나온 str

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model="google/gemma-4-31b-it:free",
        messages=messages
    )

    answer = response.choices[0].message.content

    return answer

## End to End Test

In [ ]:
!pip install -q openai

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
from openai import OpenAI
from google.colab import userdata

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list


def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기

        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results



def format_context(retrieved_chunks):
    formatted_list = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        text = chunk["text"]
        metadata = chunk["metadata"]

        source = metadata["source"]
        chunk_index = metadata["chunk_index"]

        formatted_text = (
            f"[{i}]\n"
            f"Source: {source}\n"
            f"Chunk: {chunk_index}\n"
            f"Text: {text}"
        )

        formatted_list.append(formatted_text)

    context = "\n\n".join(formatted_list)

    return context

def build_prompt(question, context):
    prompt = (
        "Answer the question using the provided context.\n\n"
        "If the context does not contain enough information, "
        "say that the information is not available in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    return prompt



api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def generate_answer(prompt): # prompt는 format_context를 거쳐서 나온 retrieved를 다시 build_prompt로 다듬어서 나온 str

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=messages
    )

    answer = response.choices[0].message.content
    model_used = response.model

    return answer, model_used

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "How does attention turn each token into a contextual representation?"

chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(chunk_embeddings.shape)
print(query_embedding.shape)


retrieved = retrieve(
    query_embedding,
    chunk_embeddings,
    all_chunks,
    k=3
)

context = format_context(retrieved)

prompt = build_prompt(query_text, context)
print(prompt)

answer, model_used = generate_answer(prompt)
print(answer)
print("Model used:", model_used)



torch.Size([153, 384])
torch.Size([384])
Answer the question using the provided context.

If the context does not contain enough information, say that the information is not available in the context.

Context:
[1]
Source: 07_attention.md
Chunk: 40
Text: Query로 다른 모든 token의 Key와 비교하고, > 해당 Value들을 가중합하면서 각 token을 문맥이 반영된 contextual representation으로 바꾼다. ### Multi-Head Attention > 같은 입력을 서로 다른 learned projection space에서 여러 번 바라보면서 > 서로 다른 attention pattern과 관계를 병렬로 학습한다. 따라서 전체적인 발전 흐름은, ```text RNN Seq2Seq ↓ Fixed context bottleneck ↓ Encoder-Decoder Attention ↓ Query / Key / Value ↓ Self-Attention ↓ Scaled Dot-Product Attention ↓ Multi-Head Attention ``` 으로 정리할 수 있다. --- ## 25. Where This Note Stops 이 문서는 **Multi-Head Self-Attention의 output을 만드는 과정까지**를 다룬다. 다음 단계에서는 이 Multi-Head Attention output이 Transformer Encoder Block 안에서 어떻게 사용되는지

[2]
Source: 09_transformer_decoder.md
Chunk: 41
Text: target position 특정 마지막 token에서만 source를 보는 것이 아니다. ### Decoder is Q, Encoder is K/V in Cross-Att

## Test without RAG

In [ ]:
!pip install -q openai

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
from openai import OpenAI
from google.colab import userdata

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list


def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기

        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results



def format_context(retrieved_chunks):
    formatted_list = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        text = chunk["text"]
        metadata = chunk["metadata"]

        source = metadata["source"]
        chunk_index = metadata["chunk_index"]

        formatted_text = (
            f"[{i}]\n"
            f"Source: {source}\n"
            f"Chunk: {chunk_index}\n"
            f"Text: {text}"
        )

        formatted_list.append(formatted_text)

    context = "\n\n".join(formatted_list)

    return context

def build_prompt(question, context):
    prompt = (
        "Answer the question using the provided context.\n\n"
        "If the context does not contain enough information, "
        "say that the information is not available in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    return prompt



api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def generate_answer(prompt): # prompt는 format_context를 거쳐서 나온 retrieved를 다시 build_prompt로 다듬어서 나온 str

    messages = [
        {
            "role": "user",
            "content": "What is the code name for the attention retrieval experiment?"
        }
    ]

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=messages
    )

    answer = response.choices[0].message.content
    model_used = response.model

    return answer, model_used

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md",
    "notes/rag_test.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "What is the code name for the attention retrieval experiment?"

chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(chunk_embeddings.shape)
print(query_embedding.shape)


retrieved = retrieve(
    query_embedding,
    chunk_embeddings,
    all_chunks,
    k=3
)

context = format_context(retrieved)

prompt = build_prompt(query_text, context)
print(prompt)

answer, model_used = generate_answer(prompt)
print(answer)
print("Model used:", model_used)



torch.Size([154, 384])
torch.Size([384])
Answer the question using the provided context.

If the context does not contain enough information, say that the information is not available in the context.

Context:
[1]
Source: rag_test.md
Chunk: 0
Text: # Mini RAG Test In this project, the code name for the attention retrieval experiment is "Purple Mango 731". The experiment was created only for testing whether the RAG system actually uses retrieved context.

[2]
Source: 07_attention.md
Chunk: 5
Text: --- ## 3. Basic Attention Calculation Attention의 기본 계산은 크게 네 단계로 볼 수 있다. ### Step 1. Score 현재 상태와 각 후보 정보 사이의 관련성을 계산한다. $$ e_i = score(query, key_i) $$ Encoder-Decoder Attention의 기본 형태로 쓰면, $$ e_{t,i} = score(s_t, h_i) $$ 이다. 여기서 `score`는 반드시 dot product일 필요는 없다. Attention의 핵심은 **관련성을 나타내는 scalar score를 만드는 것**이고, 그 방법으로 dot product, additive attention 등 여러 방식이 존재할 수 있다. --- ### Step 2. Softmax score들을 softmax에 넣어 합이 1인 attention weight로 바꾼다. $$ \alpha_i = \frac{\exp(e_i)} {\sum_j \exp(e_j)} 

## Test with RAG

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
from openai import OpenAI
from google.colab import userdata

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list


def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기

        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results



def format_context(retrieved_chunks):
    formatted_list = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        text = chunk["text"]
        metadata = chunk["metadata"]

        source = metadata["source"]
        chunk_index = metadata["chunk_index"]

        formatted_text = (
            f"[{i}]\n"
            f"Source: {source}\n"
            f"Chunk: {chunk_index}\n"
            f"Text: {text}"
        )

        formatted_list.append(formatted_text)

    context = "\n\n".join(formatted_list)

    return context

def build_prompt(question, context):
    prompt = (
        "Answer the question using the provided context.\n\n"
        "If the context does not contain enough information, "
        "say that the information is not available in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    return prompt



api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def generate_answer(prompt): # prompt는 format_context를 거쳐서 나온 retrieved를 다시 build_prompt로 다듬어서 나온 str

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=messages
    )

    answer = response.choices[0].message.content
    model_used = response.model

    return answer, model_used

In [ ]:
file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md",
    "notes/rag_test.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

model = SentenceTransformer("all-MiniLM-L6-v2")
query_text = "What is the code name for the attention retrieval experiment?"

chunk_texts = [chunk["text"] for chunk in all_chunks] # all_chunks에 있는 chunk dictionary에서 text부분만 가져와서 리스트로 담기

chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
query_embedding = model.encode(query_text, convert_to_tensor=True)

print(chunk_embeddings.shape)
print(query_embedding.shape)


retrieved = retrieve(
    query_embedding,
    chunk_embeddings,
    all_chunks,
    k=3
)

context = format_context(retrieved)

prompt = build_prompt(query_text, context)
print(prompt)

answer, model_used = generate_answer(prompt)
print(answer)
print("Model used:", model_used)



torch.Size([154, 384])
torch.Size([384])
Answer the question using the provided context.

If the context does not contain enough information, say that the information is not available in the context.

Context:
[1]
Source: rag_test.md
Chunk: 0
Text: # Mini RAG Test In this project, the code name for the attention retrieval experiment is "Purple Mango 731". The experiment was created only for testing whether the RAG system actually uses retrieved context.

[2]
Source: 07_attention.md
Chunk: 5
Text: --- ## 3. Basic Attention Calculation Attention의 기본 계산은 크게 네 단계로 볼 수 있다. ### Step 1. Score 현재 상태와 각 후보 정보 사이의 관련성을 계산한다. $$ e_i = score(query, key_i) $$ Encoder-Decoder Attention의 기본 형태로 쓰면, $$ e_{t,i} = score(s_t, h_i) $$ 이다. 여기서 `score`는 반드시 dot product일 필요는 없다. Attention의 핵심은 **관련성을 나타내는 scalar score를 만드는 것**이고, 그 방법으로 dot product, additive attention 등 여러 방식이 존재할 수 있다. --- ### Step 2. Softmax score들을 softmax에 넣어 합이 1인 attention weight로 바꾼다. $$ \alpha_i = \frac{\exp(e_i)} {\sum_j \exp(e_j)} 

## Final End-to-End Product

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
from openai import OpenAI
from google.colab import userdata

def cosine_similarity_batch(query, documents): # query: [D], documents: [N,D]

    dot_products = documents @ query # dot_products: [N]
    q_norm = torch.norm(query) # a scalar
    doc_norm = torch.norm(documents, dim=1) # doc_norm: [N]
    scores = dot_products / (q_norm * doc_norm) # element-wise division
    return scores


def load_document(file_path): # ex) data/transformer_decoder.md

  path = Path(file_path)
  text = path.read_text(encoding="utf-8")

  document_dict = {
  "text": text,
  "metadata": {
      "source": path.name,
      "path": str(path)
  }
}
  return document_dict # 문서 내용과 metadata를 둘다 가지는 dict를 리턴


def chunk_text_overlap(text, chunk_size, overlap):
  # text는 그냥 긴 str
  assert 0 <= overlap < chunk_size
  results = []
  words = text.split() # 공백기준으로 잘라서 list 반환
  for i in range(0, len(words), chunk_size-overlap):
    chunk = words[i:i + chunk_size]
    result = " ".join(chunk) # 공백 넣어서 붙이기
    results.append(result)
  return results # list of str



def chunk_document(document, chunk_size, overlap):
    # document는 text와 metadata를 같이 가지는 dictionary

    text_chunks = chunk_text_overlap(
        document["text"],
        chunk_size,
        overlap
    )

    chunks_with_metadata = []

    for i, chunk in enumerate(text_chunks):

        metadata = document["metadata"].copy()
        metadata["chunk_index"] = i

        chunk_dict = {
          "text": chunk,
          "metadata": metadata
        }
        chunks_with_metadata.append(chunk_dict)

    return chunks_with_metadata # chunk내용과 metadata를 둘다 가지는 dict들을 리스트에 넣어서 리턴


def load_and_chunk_files(file_paths, chunk_size, overlap):

    all_chunks = []

    for file_path in file_paths:

        # document 로드
        document = load_document(file_path) # 로드해서 text와 metadata를 가지는 dictionary 리턴

        # document를 chunks로 변환
        chunks = chunk_document(document, chunk_size, overlap) # chunk내용과 metadata를 둘다 가지는 dict들을 리스트로 리턴

        # all_chunks에 chunks를 합치기
        all_chunks.extend(chunks)
        # chunks는 list라서 그 안의 값들을 append()하면 []에 감싸진 형태로 들어가기 때문에,
        # extend()로 []를 벗겨서 all_chunks에 넣기.

    return all_chunks # chunks 리스트의 값들(chunk dict들)을 모두 가지는 list


def retrieve(query_embedding, chunk_embeddings, chunks, k=2):# 여기서 chunks는 all_chunks. 즉 chunk_text와 metadata를 가진 dictionary를 가진 리스트

    scores = cosine_similarity_batch(query_embedding, chunk_embeddings)

    top_scores, top_indices = torch.topk(scores, k)

    results = []

    for score, idx in zip(top_scores, top_indices):

        index = idx.item()

        # 1. chunks[index]로 원래 chunk dictionary 찾기
        chunk_dict = chunks[index]

        # 2. result dictionary 만들기

        result_dict = {
          "text": chunk_dict["text"],
          "score": score.item(),
          "metadata": chunk_dict["metadata"]
        }

        # 3. results에 append
        results.append(result_dict)

    return results



def format_context(retrieved_chunks):
    formatted_list = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        text = chunk["text"]
        metadata = chunk["metadata"]

        source = metadata["source"]
        chunk_index = metadata["chunk_index"]

        formatted_text = (
            f"[{i}]\n"
            f"Source: {source}\n"
            f"Chunk: {chunk_index}\n"
            f"Text: {text}"
        )

        formatted_list.append(formatted_text)

    context = "\n\n".join(formatted_list)

    return context

def build_prompt(question, context):
    prompt = (
        "Answer the question using the provided context.\n\n"
        "If the context does not contain enough information, "
        "say that the information is not available in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    return prompt



api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def generate_answer(prompt): # prompt는 format_context를 거쳐서 나온 retrieved를 다시 build_prompt로 다듬어서 나온 str

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=messages
    )

    answer = response.choices[0].message.content
    model_used = response.model

    return answer, model_used



def rag_answer(question, k=3): # 질문이 들어올 때마다 필요한 부분만 실행하는 함수


    # 1. question embedding
    query_embedding = embedding_model.encode(question, convert_to_tensor=True)

    # 2. retrieval
    retrieved = retrieve(
        query_embedding,
        chunk_embeddings,
        all_chunks,
        k=k
    )

    # 3. retrieved chunks → context
    context = format_context(retrieved)

    # 4. context + question → prompt
    prompt = build_prompt(question, context)

    # 5. LLM generation
    answer, model_used = generate_answer(prompt)

    return answer, model_used

In [ ]:
# 문서준비/인덱싱 단계 - 한번만 실행

file_paths = [
    "notes/07_attention.md",
    "notes/08_transformer_encoder.md",
    "notes/09_transformer_decoder.md"
]

all_chunks = load_and_chunk_files(
    file_paths,
    chunk_size=100,
    overlap=20
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_tensor=True
)


In [ ]:
# 질문 처리 단계 — 질문할 때마다 실행

question = "How does attention turn each token into a contextual representation?"

answer, model_used = rag_answer(question, 3) # 한번에 모든 과정을 묶어둬서 깔끔하게 실행 가능하게 함

print(answer)
print("Model used:", model_used)

Attention turns each token into a contextual representation by having the **Query** of a token compare with the **Keys** of all other tokens, and then taking a **weighted sum of the corresponding Values** based on those comparison scores. This process allows each token to aggregate information from other tokens in the sequence, resulting in a representation that reflects the surrounding context.
Model used: nvidia/nemotron-3-ultra-550b-a55b:free
